# End of week 1 exercise by Uriel!

Build a tool that takes a technical question,
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

**Using OpenAI API and Ollama**



In [ ]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import requests

# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

System_prompt = """
You are a helpful assistant that can explain code. assume the user is a beginner. show the code in markdown. do it as short as possible.
"""
user_prompt_question = """
Please explain what this code does and why: first write the model you are using such as "using gpt-4o-mini", then write the answer.
yield from {book.get("author") for book in books if book.get("author")}
"""

messages = [
    {"role": "system", "content": System_prompt},
    {"role": "user", "content": user_prompt_question}
]

def streaming_response(model, messages, openai_client):
    """Display streaming response from the specified client/model in Jupyter markdown."""
    stream = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

def get_gpt4o_response(messages):
    """Get and display response from OpenAI gpt-4o-mini."""
    openai = OpenAI()
    streaming_response(MODEL_GPT, messages, openai)

def get_llama_response(messages):
    """Get and display response from local Ollama Llama3.2."""
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    streaming_response(MODEL_LLAMA, messages, ollama)

def main():
    load_dotenv(override=True)
    api_key = os.getenv('OPENAI_API_KEY')

    if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
        print("API key looks good so far")
    else:
        print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

    print("\nAnswer from GPT-4o-mini:")
    get_gpt4o_response(messages)
    print("\nAnswer from Llama 3.2 (Ollama):")
    get_llama_response(messages)

main()
    